# Lab 2: Gradient Descent from Scratch

Gradient descent is the algorithm that makes neural network training possible. The update rule $x \leftarrow x - \eta \cdot f'(x)$ is derived directly from the linear approximation: stepping opposite the gradient decreases the function locally.

In this lab you implement it from scratch — no optimizers — and build intuition for why learning rate is the most sensitive hyperparameter.

**Sections**
1. 1D Gradient Descent
2. Learning Rate Experiments
3. 2D Gradient Descent on a Bowl
4. Linear Regression via Gradient Descent
5. Comparison with `torch.optim.SGD`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
rng = np.random.default_rng(42)
print('Setup complete.')

## 1. 1D Gradient Descent

We minimize $f(x) = x^2 - 4x + 5$. By completing the square: $f(x) = (x-2)^2 + 1$, so the minimum is at $x=2$, $f(2)=1$.

The derivative is $f'(x) = 2x - 4$. Starting at $x_0 = 0$, each step is $x \leftarrow x - \eta(2x-4)$.

In [ ]:
def f(x):     return x**2 - 4*x + 5
def f_prime(x): return 2*x - 4

def gradient_descent_1d(x0, lr, n_steps):
    x = x0
    history = [x]
    for _ in range(n_steps):
        x = x - lr * f_prime(x)
        history.append(x)
    return np.array(history)

hist = gradient_descent_1d(x0=0.0, lr=0.1, n_steps=30)

x_plot = np.linspace(-0.5, 4.5, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: trajectory on the function curve
ax1.plot(x_plot, f(x_plot), 'steelblue', lw=2, label='f(x)')
ax1.scatter(hist, f(hist), c=range(len(hist)), cmap='plasma', s=40, zorder=5)
ax1.plot(hist, f(hist), 'k--', lw=0.8, alpha=0.5)
ax1.scatter([2], [1], marker='*', s=200, color='gold', zorder=6, label='minimum (2, 1)')
ax1.set_xlabel('x')
ax1.set_title('Trajectory on f(x)')
ax1.legend()

# Right: convergence curves
ax2.plot(f(hist), 'coral', lw=2)
ax2.axhline(1.0, color='gray', linestyle=':', label='f(x*)=1')
ax2.set_xlabel('Step')
ax2.set_ylabel('f(x)')
ax2.set_title('Loss over steps (lr=0.1)')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final x = {hist[-1]:.6f}  (target: 2.0)")
print(f"Final f(x) = {f(hist[-1]):.6f}  (target: 1.0)")

## 2. Learning Rate Experiments

The learning rate $\eta$ determines whether GD converges smoothly, converges slowly, or diverges entirely.

In [ ]:
configs = [
    ('Too small  η=0.01', 0.01),
    ('Just right η=0.3',  0.30),
    ('Too large  η=1.05', 1.05),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (label, lr) in zip(axes, configs):
    hist = gradient_descent_1d(x0=0.0, lr=lr, n_steps=40)
    losses = f(hist)
    losses = np.clip(losses, None, 200)  # clip for display
    ax.plot(losses, lw=2, color='steelblue')
    ax.axhline(1.0, color='gray', linestyle=':', alpha=0.7)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Step')
    if ax == axes[0]:
        ax.set_ylabel('f(x)')

plt.suptitle('Effect of learning rate on convergence', y=1.02)
plt.tight_layout()
plt.show()

print("η=0.01: converges but takes many steps (tangent barely moves)")
print("η=0.30: converges quickly and smoothly")
print("η=1.05: overshoots — loss oscillates or diverges")

## 3. 2D Gradient Descent on a Bowl

In multiple dimensions, the gradient is a vector of partial derivatives. We minimize $f(x,y) = x^2 + y^2$ (bowl centered at origin) with update $(x,y) \leftarrow (x,y) - \eta\,\nabla f$, where $\nabla f = (2x,\, 2y)$.

In [ ]:
def f2d(x, y):   return x**2 + y**2
def grad2d(x, y): return np.array([2*x, 2*y])

def gd_2d(start, lr, n_steps):
    pos = np.array(start, dtype=float)
    path = [pos.copy()]
    for _ in range(n_steps):
        pos -= lr * grad2d(*pos)
        path.append(pos.copy())
    return np.array(path)

path = gd_2d(start=[3.0, -2.5], lr=0.15, n_steps=25)

# Contour plot
grid = np.linspace(-3.5, 3.5, 200)
X, Y = np.meshgrid(grid, grid)
Z = f2d(X, Y)

plt.figure(figsize=(7, 6))
plt.contourf(X, Y, Z, levels=20, cmap='Blues', alpha=0.7)
plt.contour(X, Y, Z, levels=20, colors='white', linewidths=0.5, alpha=0.4)
plt.plot(path[:, 0], path[:, 1], 'o-', color='coral', lw=1.5, ms=5, label='GD path')
plt.scatter([path[0, 0]], [path[0, 1]], color='yellow', s=100, zorder=5, label='start')
plt.scatter([0], [0], marker='*', color='gold', s=200, zorder=6, label='minimum (0,0)')
plt.colorbar(label='f(x,y)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Gradient descent on f(x,y) = x² + y²')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Linear Regression via Gradient Descent

We fit $y = wx + b$ to noisy data by minimizing MSE $\mathcal{L} = \frac{1}{N}\sum_i(wx_i + b - y_i)^2$.

The gradients are:
$$\frac{\partial\mathcal{L}}{\partial w} = \frac{2}{N}\sum_i x_i(wx_i+b-y_i), \qquad \frac{\partial\mathcal{L}}{\partial b} = \frac{2}{N}\sum_i (wx_i+b-y_i)$$

In [ ]:
# Generate noisy data from y = 2x + 1
N = 80
x_data = rng.uniform(-3, 3, N)
y_data = 2.0 * x_data + 1.0 + rng.normal(0, 0.8, N)

def mse(w, b, x, y):
    preds = w * x + b
    return np.mean((preds - y)**2)

def mse_grads(w, b, x, y):
    preds = w * x + b
    residuals = preds - y
    dw = 2 * np.mean(x * residuals)
    db = 2 * np.mean(residuals)
    return dw, db

# Train
w, b = 0.0, 0.0
lr = 0.05
losses = []

for step in range(200):
    losses.append(mse(w, b, x_data, y_data))
    dw, db = mse_grads(w, b, x_data, y_data)
    w -= lr * dw
    b -= lr * db

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(x_data, y_data, alpha=0.5, s=20, label='data')
x_line = np.array([-3, 3])
ax1.plot(x_line, 2*x_line + 1, 'gray', linestyle=':', lw=2, label='true: y=2x+1')
ax1.plot(x_line, w*x_line + b, 'coral', lw=2, label=f'fit: y={w:.2f}x+{b:.2f}')
ax1.set_title('Fitted line')
ax1.legend(fontsize=9)

ax2.plot(losses, 'steelblue', lw=2)
ax2.set_xlabel('Step')
ax2.set_ylabel('MSE')
ax2.set_title('Training loss')

plt.tight_layout()
plt.show()

print(f"Recovered: w={w:.4f} (true 2.0), b={b:.4f} (true 1.0)")

## 5. Comparison with `torch.optim.SGD`

Let's confirm our hand-rolled GD matches PyTorch's `SGD` optimizer step-for-step.

In [ ]:
x_t = torch.tensor(x_data, dtype=torch.float32)
y_t = torch.tensor(y_data, dtype=torch.float32)

w_pt = torch.tensor(0.0, requires_grad=True)
b_pt = torch.tensor(0.0, requires_grad=True)
optimizer = optim.SGD([w_pt, b_pt], lr=0.05)

torch_losses = []
for _ in range(200):
    optimizer.zero_grad()
    preds = w_pt * x_t + b_pt
    loss = torch.mean((preds - y_t)**2)
    torch_losses.append(loss.item())
    loss.backward()
    optimizer.step()

plt.figure(figsize=(8, 4))
plt.plot(losses, lw=2, label='Hand-rolled GD', color='steelblue')
plt.plot(torch_losses, lw=2, linestyle='--', label='torch.optim.SGD', color='coral')
plt.xlabel('Step')
plt.ylabel('MSE')
plt.title('Hand-rolled GD vs torch.optim.SGD (should be identical)')
plt.legend()
plt.tight_layout()
plt.show()

max_diff = max(abs(a - b) for a, b in zip(losses, torch_losses))
print(f"Max loss difference between implementations: {max_diff:.2e}")
print("Effectively zero — they are the same algorithm.")